# Uber Fare Prediction

**Objective:**
1. Understand the dataset & cleanup (if required).
2. Build regression models to predict the fare price of Uber rides.
3. Evaluate and compare models using R², RMSE, and MAE.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

## 1. Understand the Dataset

In [ ]:
df = pd.read_csv("uber.csv")
print("Dataset Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print("Missing values:\n", df.isnull().sum())

## 2. Data Cleanup & Feature Engineering

In [ ]:
# Drop identifiers and missing values
df = df.drop(columns=["Unnamed: 0", "key"]).dropna()

# Filter invalid coordinates and unrealistic values (NYC bounding box & standard ranges)
df = df[
    df["pickup_longitude"].between(-74.3, -73.7)
    & df["dropoff_longitude"].between(-74.3, -73.7)
    & df["pickup_latitude"].between(40.5, 41.0)
    & df["dropoff_latitude"].between(40.5, 41.0)
    & df["passenger_count"].between(1, 6)
    & df["fare_amount"].between(2.5, 200)
]

In [ ]:
# Calculate Haversine distance in km
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

df["distance_km"] = haversine(
    df["pickup_latitude"], df["pickup_longitude"],
    df["dropoff_latitude"], df["dropoff_longitude"]
)

# Extract datetime features and drop original timestamp
dt = pd.to_datetime(df["pickup_datetime"])
df["hour"] = dt.dt.hour
df["day"] = dt.dt.day
df["month"] = dt.dt.month
df["year"] = dt.dt.year
df["day_of_week"] = dt.dt.dayofweek

# Filter out zero or extreme trip distances and drop datetime
df = df[df["distance_km"].between(0.1, 100)]
df = df.drop(columns=["pickup_datetime"])
df.head()

## 3. Train / Test Split

In [ ]:
X = df.drop(columns=["fare_amount"])
y = df["fare_amount"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)

## 4. Build Regression Models

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    results.append({
        "Model": name,
        "R2 Score": r2_score(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "MAE": mean_absolute_error(y_test, y_pred)
    })

comparison_df = pd.DataFrame(results)
comparison_df

## 5. Model Evaluation & Comparison

In [ ]:
comparison_df.set_index("Model")[["RMSE", "MAE"]].plot(kind="bar", figsize=(8, 4))
plt.title("Model Error Comparison (Lower is Better)")
plt.ylabel("USD ($)")
plt.xticks(rotation=0)
plt.show()